#### Capacidad instalada de generación distribuida en la Ciudad de México

In [1]:
rm(list=ls()) 

In [2]:
if(!require('pacman')) install.packages('pacman')
pacman::p_load(tidyverse, showtext, srvyr,httr,utils,foreign, ggplot2)
remotes::install_github("cpyp-ipdp/funcionespgd", force = TRUE)
library(funcionespgd)
font_add_google("Poppins", "pop")
showtext_auto()

Cargando paquete requerido: pacman




rlang    (1.1.7 -> 1.3.0) [CRAN]
glue     (1.8.0 -> 1.8.1) [CRAN]
cli      (3.6.5 -> 3.6.6) [CRAN]
vctrs    (0.7.1 -> 0.7.3) [CRAN]
magrittr (2.0.4 -> 2.0.5) [CRAN]
S7       (0.2.1 -> 0.2.2) [CRAN]
dplyr    (1.2.0 -> 1.2.1) [CRAN]
ggplot2  (4.0.2 -> 4.0.3) [CRAN]


Installing 8 packages: rlang, glue, cli, vctrs, magrittr, S7, dplyr, ggplot2

Warning message:
"packages 'dplyr', 'ggplot2' are in use and will not be installed"
Installing packages into 'C:/Users/IPDP 2026/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



package 'rlang' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'rlang'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problema al copiar C:\Users\IPDP 2026\AppData\Local\R\win-library\4.5\00LOCK\rlang\libs\x64\rlang.dll  a C:\Users\IPDP 2026\AppData\Local\R\win-library\4.5\rlang\libs\x64\rlang.dll: Permission denied"
Warning message:
"restored 'rlang'"


package 'glue' successfully unpacked and MD5 sums checked


... (9 outputs left out)


The downloaded binary packages are in
	C:\Users\IPDP 2026\AppData\Local\Temp\RtmpEhAwRf\downloaded_packages
── R CMD build ─────────────────────────────────────────────────────────────────



Please download and install Rtools 4.5 from https://cran.r-project.org/bin/windows/Rtools/.



* checking for file 'C:\Users\IPDP 2026\AppData\Local\Temp\RtmpEhAwRf\remotes2e182736be\cpyp-ipdp-funcionespgd-6fd64b9/DESCRIPTION' ... OK
* preparing 'funcionespgd':
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
Omitted 'LazyData' from DESCRIPTION
* building 'funcionespgd_0.1.0.tar.gz'



Installing package into 'C:/Users/IPDP 2026/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



In [3]:
# DATOS: capacidad instalada de generación distribuida en la Ciudad de México
datos <- read_csv("capacidad_instalada_distribuida.csv")

datos

Rows: 3 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (2): year, capacidad_instalada_mw

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


year,capacidad_instalada_mw
<dbl>,<dbl>
2022,126.59
2023,148.07
2024,177.67


In [4]:
dato_final <- datos %>%
  filter(year == 2024) %>%
  select(capacidad_instalada_mw) %>%
    pull()
print("Valor final")
dato_final
dato_inicial <- datos %>%
  filter(year == 2022) %>%
  select(capacidad_instalada_mw) %>%
    pull()
print("Valor inicial")
dato_inicial

tcma <- (( dato_final / dato_inicial)^(1/(2024-2022)) - 1)
print("Tasa de crecimiento anual promedio")
tcma

[1] "Valor final"


[1] 177.67

[1] "Valor inicial"


[1] 126.59

[1] "Tasa de crecimiento anual promedio"


[1] 0.1846972

In [5]:
tcma = 0.12

In [23]:
extendido <- data.frame(year = 2025:2047)
extendido$capacidad_instalada_mw <- datos$capacidad_instalada_mw[datos$year == 2024] *
  cumprod(rep(1 + tcma, length(2025:2047)))

extendido <- extendido %>%
  mutate(capacidad_instalada_mw = round(capacidad_instalada_mw, 1))

extendido

year,capacidad_instalada_mw
<int>,<dbl>
2025,199.0
2026,222.9
2027,249.6
2028,279.6
2029,313.1
2030,350.7
2031,392.8
2032,439.9
2033,492.7


In [25]:
serie_completa <- dplyr::bind_rows(
    datos %>%
dplyr::mutate(year = as.numeric(year)) %>%
      intervals(margen = 0, año_base = 2023,var = "capacidad_instalada_mw"),

    extendido %>%
      dplyr::mutate(year = as.numeric(year)) %>%
      intervals(margen = 0.02, año_base = 2023,var="capacidad_instalada_mw"))


serie_completa <- serie_completa %>%
  mutate(across(where(is.numeric), ~ round(.x, 1)))
serie_completa

year,estimado,superior,inferior
<dbl>,<dbl>,<dbl>,<dbl>
2022,126.6,126.6,126.6
2023,148.1,148.1,148.1
2024,177.7,177.7,177.7
2025,199.0,207.0,191.3
2026,222.9,236.5,210.0
2027,249.6,270.2,230.6
2028,279.6,308.7,253.2
2029,313.1,352.6,278.0
2030,350.7,402.8,305.3


In [26]:
serie_completa <- serie_completa %>%
  rename(periodo = year,
         deseable = estimado,
         transformador = superior,
         inercial = inferior
         )

serie_completa

periodo,deseable,transformador,inercial
<dbl>,<dbl>,<dbl>,<dbl>
2022,126.6,126.6,126.6
2023,148.1,148.1,148.1
2024,177.7,177.7,177.7
2025,199.0,207.0,191.3
2026,222.9,236.5,210.0
2027,249.6,270.2,230.6
2028,279.6,308.7,253.2
2029,313.1,352.6,278.0
2030,350.7,402.8,305.3


In [27]:
write.csv(
  serie_completa,
  "Capacidad_energetica.csv",
  row.names = FALSE
)